### Các thuật toán dùng để align bbox khá tốt:
1. Dùng levenshtein_align_boxes:

In [2]:
import pandas as pd
import os
from dotenv import load_dotenv
from pathlib import Path
import numpy as np
load_dotenv(".env")
from align.align import *

similar = pd.read_excel(os.environ['NOM_SIMILARITY_DICTIONARY'])
trans = pd.read_excel(os.environ['QN2NOM_DICTIONARY']).iloc[:, [0,1]]



def is_compatible(han_nom_char, quoc_ngu_word, similar_df, trans_df):

    hn_candidates = trans_df[trans_df.iloc[:, 0] == quoc_ngu_word].iloc[:, 1].tolist()
    if not hn_candidates:
        return False


    similar_chars = similar_df[similar_df.iloc[:, 0] == han_nom_char].iloc[:, 1].tolist()
    similar_chars.append(han_nom_char)  # bao gồm luôn chính nó


    return bool(set(hn_candidates) & set(similar_chars))


def levenshtein_align_boxes(nom_list, qn_list, similar_df, trans_df):
    m, n = len(nom_list), len(qn_list)
    dp = np.zeros((m + 1, n + 1), dtype=int)
    backtrace = np.empty((m + 1, n + 1), dtype=object)

    for i in range(m + 1):
        dp[i][0] = i
        backtrace[i][0] = 'U'
    for j in range(n + 1):
        dp[0][j] = j
        backtrace[0][j] = 'L'

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            match = is_compatible(nom_list[i - 1], qn_list[j - 1], similar_df, trans_df)
            cost = 0 if match else 1
            options = [
                (dp[i - 1][j] + 1, 'U'),
                (dp[i][j - 1] + 1, 'L'),
                (dp[i - 1][j - 1] + cost, 'D')
            ]
            dp[i][j], backtrace[i][j] = min(options)

    aligned_nom, aligned_qn = [], []
    i, j = m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and backtrace[i][j] == 'D':
            aligned_nom.insert(0, nom_list[i - 1])
            aligned_qn.insert(0, qn_list[j - 1])
            i -= 1
            j -= 1
        elif i > 0 and backtrace[i][j] == 'U':
            aligned_nom.insert(0, nom_list[i - 1])
            aligned_qn.insert(0, "_")
            i -= 1
        elif j > 0 and backtrace[i][j] == 'L':
            aligned_nom.insert(0, "_")
            aligned_qn.insert(0, qn_list[j - 1])
            j -= 1

    return aligned_nom, aligned_qn



In [10]:
x = "積善 慶 家子 孫永 傳 苗裔 功德 員 成 願 情 如意 香 臺 立 寺 以為 代 代 錫 興 功 會 主 壇 那 諸人 等".replace(" ", "")
print(x)
nom_data =  list(x)  # hoặc dùng tokenizer nếu có
quoc_ngu_list = "tích thiện khánh gia tử tôn vĩnh truyền miêu duệ công đức viên thành nguyện tình như ý hương đài nhất tự dĩ vi đại tích hưng tích hưng công hội chủ đàn na chư nhân quyến đẳng".lower().split()
new_result = levenshtein_align_boxes(nom_data, quoc_ngu_list, similar, trans)
print(new_result[0])
print(new_result[1])

積善慶家子孫永傳苗裔功德員成願情如意香臺立寺以為代代錫興功會主壇那諸人等
['積', '善', '慶', '家', '子', '孫', '永', '傳', '苗', '裔', '功', '德', '員', '成', '願', '情', '如', '意', '香', '臺', '立', '寺', '以', '為', '代', '_', '代', '錫', '興', '功', '會', '主', '壇', '那', '諸', '人', '_', '等']
['tích', 'thiện', 'khánh', 'gia', 'tử', 'tôn', 'vĩnh', 'truyền', 'miêu', 'duệ', 'công', 'đức', 'viên', 'thành', 'nguyện', 'tình', 'như', 'ý', 'hương', 'đài', 'nhất', 'tự', 'dĩ', 'vi', 'đại', 'tích', 'hưng', 'tích', 'hưng', 'công', 'hội', 'chủ', 'đàn', 'na', 'chư', 'nhân', 'quyến', 'đẳng']


In [16]:
x = "".join(new_result[0])
vi_tri = x.find("_")
new_result[1][vi_tri]

'tích'

2. Dùng needleman_wunsch_with_lists_2:

In [17]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import ast
import pandas as pd
import os
from dotenv import load_dotenv
import numpy as np
load_dotenv(".env")
from align.align import align_boxes_2



similar = pd.read_excel(os.environ['NOM_SIMILARITY_DICTIONARY']) ### chỉnh lại địa chỉ hoặc để vào env. trong ocr_corrector
trans = pd.read_excel(os.environ['QN2NOM_DICTIONARY']).iloc[:, [0,1]]

def is_compatible(han_nom_char, quoc_ngu_word, similar_df, trans_df):
    hn_candidates = trans_df[trans_df.iloc[:, 0] == quoc_ngu_word].iloc[:, 1].tolist()
    if not hn_candidates:
        return False

    similar_chars = similar_df[similar_df.iloc[:, 0] == han_nom_char].iloc[:, 1].tolist()
    similar_chars.append(han_nom_char)
    return bool(set(hn_candidates) & set(similar_chars))

def needleman_wunsch_with_lists_2(nom_list, quoc_ngu_list, similar_df, trans_df, match_score=1, gap_penalty=-1):
    len_nom = len(nom_list)
    len_quoc_ngu = len(quoc_ngu_list)

    scores = [[0] * (len_quoc_ngu + 1) for _ in range(len_nom + 1)]
    traceback = [[None] * (len_quoc_ngu + 1) for _ in range(len_nom + 1)]

    for i in range(1, len_nom + 1):
        scores[i][0] = i * gap_penalty
        traceback[i][0] = 'U'
    for j in range(1, len_quoc_ngu + 1):
        scores[0][j] = j * gap_penalty
        traceback[0][j] = 'L'

    for i in range(1, len_nom + 1):
        for j in range(1, len_quoc_ngu + 1):
            match = is_compatible(nom_list[i - 1], quoc_ngu_list[j - 1], similar_df, trans_df)
            score_match = scores[i - 1][j - 1] + (match_score if match else -match_score)
            score_delete = scores[i - 1][j] + gap_penalty
            score_insert = scores[i][j - 1] + gap_penalty

            max_score = max(score_match, score_delete, score_insert)
            scores[i][j] = max_score

            if max_score == score_match:
                traceback[i][j] = 'D'
            elif max_score == score_delete:
                traceback[i][j] = 'U'
            else:
                traceback[i][j] = 'L'

    aligned_nom, aligned_quoc_ngu = [], []
    i, j = len_nom, len_quoc_ngu

    while i > 0 or j > 0:
        if i > 0 and j > 0 and traceback[i][j] == 'D':
            aligned_nom.insert(0, nom_list[i - 1])
            aligned_quoc_ngu.insert(0, quoc_ngu_list[j - 1])
            i -= 1
            j -= 1
        elif i > 0 and (j == 0 or traceback[i][j] == 'U'):
            aligned_nom.insert(0, nom_list[i - 1])
            aligned_quoc_ngu.insert(0, '_')
            i -= 1
        elif j > 0 and (i == 0 or traceback[i][j] == 'L'):
            aligned_nom.insert(0, '_')
            aligned_quoc_ngu.insert(0, quoc_ngu_list[j - 1])
            j -= 1

    return aligned_nom, aligned_quoc_ngu

def find_best_alignment_box_2(nom_string, quoc_ngu_list, begin_len, k, similar_df, trans_df):
    nom_length = len(nom_string)
    quoc_ngu_length = len(quoc_ngu_list)

    best_score = float('-inf')
    best_quoc_ngu_substring = None
    aligned_nom_string = None
    aligned_quoc_ngu_string = None
    end_position = None

    start_idx_range = range(begin_len, min(begin_len + k, quoc_ngu_length))

    for start_idx in start_idx_range:
        end_idx_min = start_idx + nom_length - 1 - k
        end_idx_max = start_idx + nom_length + k
        end_idx_range = range(max(start_idx, end_idx_min), min(end_idx_max, quoc_ngu_length) + 1)

        for end_idx in end_idx_range:
            quoc_ngu_chars = quoc_ngu_list[start_idx:end_idx]
            aligned_nom, aligned_quoc_ngu = needleman_wunsch_with_lists_2(
                list(nom_string), quoc_ngu_chars, similar_df, trans_df
            )
            aligned_nom_array = np.array(aligned_nom)
            aligned_quoc_ngu_array = np.array(aligned_quoc_ngu)
            gaps = (aligned_nom_array == '_') | (aligned_quoc_ngu_array == '_')
            alignment_score = np.sum(-1.5 * gaps + 1.0 * ~gaps)
            if alignment_score > best_score:
                best_score = alignment_score
                best_quoc_ngu_substring = quoc_ngu_chars
                aligned_nom_string = ''.join(aligned_nom)
                aligned_quoc_ngu_string = ' '.join(aligned_quoc_ngu)
                end_position = end_idx

    return best_quoc_ngu_substring, aligned_nom_string, aligned_quoc_ngu_string, best_score, end_position

def align_boxes_2(nom_data, quoc_ngu_list, similar_df, trans_df, k=2):
    nom_list = nom_data['text']
    aligned_results = []
    begin_len = 0
    for idx, nom_string in enumerate(nom_list):
        result = find_best_alignment_box_2(nom_string, quoc_ngu_list, begin_len, k, similar_df, trans_df)
        result += (nom_data['bbox'][idx],)
        aligned_results.append(result)
        _, _, _, _, end_position, _ = result
        if end_position is not None:
            begin_len = end_position
        else:
            break
    return aligned_results


In [ ]:
nom_data =  {"text": ["陣𩄲𦰟旗桃"],
             "bbox": [[80, 525], [264, 527], [264, 552], [80, 549]] }
quoc_ngu_list = "Trận mây theo ngọn cờ".lower().split()
new_result = align_boxes_2(nom_data, quoc_ngu_list, similar, trans)
new_result

[(['trận', 'mây', 'theo', 'ngọn', 'cờ'],
  '陣𩄲_𦰟旗桃',
  'trận mây theo ngọn cờ _',
  1.0,
  5,
  [80, 525])]

In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
from pathlib import Path
import numpy as np
load_dotenv(".env")
from align.align import align_boxes_2

similar = pd.read_excel(os.environ['NOM_SIMILARITY_DICTIONARY'])
trans = pd.read_excel(os.environ['QN2NOM_DICTIONARY']).iloc[:, [0,1]]




In [18]:
import ast
x = "艾"
y = "nghệ"
list_hn  =  list(similar[similar["Input Character"] == x]["Top 20 Similar Characters"])
list_hn = ast.literal_eval(list_hn[0])
list_qn =  trans[trans["QuocNgu"] == y.lower()]["SinoNom"].to_list()
def get_intersection_hn_qn(x_han, y_qn, similar_df, trans_df):
    try:
        list_hn = ast.literal_eval(similar_df[similar_df["Input Character"] == x_han]["Top 20 Similar Characters"].values[0])
        list_qn = trans_df[trans_df["QuocNgu"] == y_qn.lower()]["SinoNom"].to_list()
        return list(set(list_hn) & set(list_qn))
    except Exception as e:
        print("Lỗi:", e)
        return []
temp = get_intersection_hn_qn("艾", "nghệ", similar, trans)
temp

[]

In [2]:
import Levenshtein
from align.color import *

ocr = "懷蘿𠬠九眞清"
word = "Hoài hoan Nghệ Cửu chân Thanh"


word = normalize_vietnamese_text(word)
a = word.split()
b = list(ocr)

max_len = len(b)
sum_char = len(b)
temp = []
_tem_1 = []
for i in range(max_len):
    result = compare(a[i], b[i])
    if len(result) >1:
        print(result)
        print(f"{a[i]} : {b[i]} => {result[0]}")

['艾', '乂']
Nghệ : 𠬠 => 艾


In [1]:
import Levenshtein
from align.color import *

ocr = "懷蘿𠬠九眞清"
word = "Hoài hoan Nghệ Cửu chân Thanh"


word = normalize_vietnamese_text(word)
a = word.split()
b = list(ocr)

max_len = len(b)
sum_char = len(b)
temp = []
_tem_1 = []
for i in range(max_len):
    result = compare(a[i], b[i])
    if len(result) >1:
        print(f"{a[i]} : {b[i]} => {result[0]}")

[('艾', 3), ('乂', 68)]
Nghệ : 𠬠 => 艾
